# Chapter 1 Exercises 8-9

These exercises inspect the merit function directly. Exercise 8 separates the horizontal and vertical merit contributions. Exercise 9 switches one target off at a time and optimizes with only one phase-advance target active.

In [3]:
using SciBmad
using GTPSA
using LinearAlgebra
using Printf
using Optim

function find_tutorial_root()
    candidates = [
        pwd(),
        normpath(joinpath(pwd(), "..")),
        normpath(joinpath(pwd(), "..", "..")),
        normpath(joinpath(pwd(), "..", "..", "..")),
    ]
    for root in candidates
        if isfile(joinpath(root, "chapter01_fodo_scibmad.ipynb")) && isdir(joinpath(root, "lattices", "chapter_1"))
            return root
        end
    end
    error("Cannot locate Ring_Design_Tutorial_SciBmad root.")
end

const tutorial_root = find_tutorial_root()
include(joinpath(tutorial_root, "lattices", "chapter_1", "chapter1_fodoF_solution.jl"))

const L_quad = 0.5
const D1_len = 0.609
const D2_len = 1.241
const B_chord = 6.86
const B_angle = pi / 132
const B_arc = B_chord * B_angle / (2sin(B_angle / 2))

const species_ref = Species("electron")
const E_ref = 18e9

D1() = Drift(name="D1", L=D1_len)
D2() = Drift(name="D2", L=D2_len)
B(name="B") = SBend(name=name, L=B_arc, angle=B_angle)

function build_forward_arc_fodo(kqf, kqd)
    return Beamline(
        [
            Quadrupole(name="QF", L=L_quad, Kn1=kqf), D1(), B("B1"), D2(),
            Quadrupole(name="QD", L=L_quad, Kn1=kqd), D1(), B("B2"), D2(),
        ];
        species_ref=species_ref,
        E_ref=E_ref,
    )
end


build_forward_arc_fodo (generic function with 1 method)

## Helper Functions

We compute the one-cell transfer matrix directly and extract the two transverse phase advances. No new derivative or optimization machinery is needed here.

In [1]:
function track_a_particle(v0, beamline)
    v = similar(v0)
    v .= v0
    bunch = Bunch(v; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    track!(bunch, beamline)
    return copy(bunch.coords.v)
end

function linear_map(beamline; x0=zeros(6))
    d = Descriptor(6, 2)
    xvars = vars(d)
    v0 = [x0[i] + xvars[i] for i in 1:6]
    vout = track_a_particle(v0, beamline)

    M = zeros(6, 6)
    for i in 1:6, j in 1:6
        powers = zeros(Int, 6)
        powers[j] = 1
        M[i, j] = vout[i][powers]
    end
    return M
end

function stable_phase_advance(M2)
    cos_mu = clamp(0.5 * tr(M2), -1.0, 1.0)
    return acos(cos_mu)
end

function cell_phase_advances(k)
    M = linear_map(build_forward_arc_fodo(k[1], k[2]))
    mu_x = stable_phase_advance(M[1:2, 1:2])
    mu_y = stable_phase_advance(M[3:4, 3:4])
    return mu_x, mu_y
end


cell_phase_advances (generic function with 1 method)

## Exercise 8: Inspect Individual Merit Contributions

The total merit is the sum of the horizontal and vertical contributions,

$$
M = M_x + M_y,
\qquad
M_x = w_x\left(\mu_x-\frac{\pi}{2}\right)^2,
\qquad
M_y = w_y\left(\mu_y-\frac{\pi}{2}\right)^2.
$$

We print the two pieces before and after the full two-plane optimization.

In [4]:
function merit_terms(k; wx=1.0, wy=1.0)
    mu_x, mu_y = cell_phase_advances(k)
    Mx = wx * (mu_x - pi / 2)^2
    My = wy * (mu_y - pi / 2)^2
    return Mx, My
end

function merit(k; use_x=true, use_y=true, wx=1.0, wy=1.0)
    Mx, My = merit_terms(k; wx=wx, wy=wy)
    return (use_x ? Mx : 0.0) + (use_y ? My : 0.0)
end

function print_phase_and_merit(label, k; use_x=true, use_y=true)
    mu_x, mu_y = cell_phase_advances(k)
    Mx, My = merit_terms(k)
    println(label)
    @printf("  KQF = %+.15f, KQD = %+.15f\n", k[1], k[2])
    @printf("  mu_x = %.12f deg, mu_y = %.12f deg\n", rad2deg(mu_x), rad2deg(mu_y))
    @printf("  Mx = %.12e, My = %.12e, active merit = %.12e\n", Mx, My, merit(k; use_x=use_x, use_y=use_y))
end

k0 = [0.4, -0.4]
k_full = [kQF_arc, kQD_arc]
lower = [0.2, -0.6]
upper = [0.6, -0.2]
print_phase_and_merit("Before optimization:", k0)
println()
print_phase_and_merit("After full two-plane optimization:", k_full)

Mx0, My0 = merit_terms(k0)
dominant = Mx0 > My0 ? "horizontal" : "vertical"
println()
println("Initially, the larger merit contribution is from the ", dominant, " plane.")


Before optimization:
  KQF = +0.400000000000000, KQD = -0.400000000000000
  mu_x = 129.493933088923 deg, mu_y = 129.417492966727 deg
  Mx = 4.751333415773e-01, My = 4.732958895904e-01, active merit = 9.484292311678e-01

After full two-plane optimization:
  KQF = +0.312659097022843, KQD = -0.312792879287368
  mu_x = 90.000000018742 deg, mu_y = 90.000000001744 deg
  Mx = 1.069975421367e-19, My = 9.263560667179e-22, active merit = 1.079238982034e-19

Initially, the larger merit contribution is from the horizontal plane.


## Exercise 9: Enable and Disable a Target

Now we switch one phase target off at a time. With only one active target, there is only one scalar condition but two adjustable quadrupole strengths. Therefore the solution is not unique: the optimizer can find one point on a curve of possible solutions, and the inactive plane is generally not controlled.

In [5]:
result_x_only = optimize(
    k -> merit(k; use_x=true, use_y=false),
    lower,
    upper,
    copy(k0),
    Fminbox(LBFGS()),
    Optim.Options(iterations=100, show_trace=false),
)

result_y_only = optimize(
    k -> merit(k; use_x=false, use_y=true),
    lower,
    upper,
    copy(k0),
    Fminbox(LBFGS()),
    Optim.Options(iterations=100, show_trace=false),
)

k_x_only = Optim.minimizer(result_x_only)
k_y_only = Optim.minimizer(result_y_only)

print_phase_and_merit("Optimized with only the horizontal target active:", k_x_only; use_x=true, use_y=false)
println()
print_phase_and_merit("Optimized with only the vertical target active:", k_y_only; use_x=false, use_y=true)
println()
print_phase_and_merit("For comparison, optimized with both targets active:", k_full; use_x=true, use_y=true)


Optimized with only the horizontal target active:
  KQF = +0.328042692538155, KQD = -0.411376611845893
  mu_x = 90.000000011684 deg, mu_y = 139.174441676466 deg
  Mx = 4.158173309862e-20, My = 7.366032157771e-01, active merit = 4.158173309862e-20

Optimized with only the vertical target active:
  KQF = +0.411342225938096, KQD = -0.328186978456784
  mu_x = 139.225897811789 deg, mu_y = 90.000000011624 deg
  Mx = 7.381455855196e-01, My = 4.115855623440e-20, active merit = 4.115855623440e-20

For comparison, optimized with both targets active:
  KQF = +0.312659097022843, KQD = -0.312792879287368
  mu_x = 90.000000018742 deg, mu_y = 90.000000001744 deg
  Mx = 1.069975421367e-19, My = 9.263560667179e-22, active merit = 1.079238982034e-19
